# Dataset Explorer

Интерактивный обзор всех собранных датасетов:

- **CCPD2019** — Китай, 4 угла + bbox (зашиты в имя файла)
- **Roboflow Russian / European / Generic** — bbox-only, YOLO формат
- **OpenALPR Benchmark** — bbox-only, разделено по br/eu/us

Ниже — виджет: выбираешь датасет, жмёшь **🎲 Обновить** — рисуется случайная сетка с разметкой.

Для CCPD рисуются и bbox (magenta), и 4 угла (TL=red, TR=green, BR=blue, BL=yellow) с полигоном.
Для остальных — только bbox.


In [ ]:
from __future__ import annotations

import random
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import ipywidgets as widgets
from IPython.display import display, clear_output

# Font fallback: латиница из Helvetica/Arial, китайские провинции CCPD —
# из Hiragino Sans GB / STHeiti (встроены на macOS).
plt.rcParams["font.sans-serif"] = [
    "Helvetica", "Arial",
    "Hiragino Sans GB", "STHeiti", "Apple SD Gothic Neo",  # CJK fallback
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

# Подключаем наши парсеры из src/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from plates.ccpd import parse_filename as parse_ccpd_filename
from plates.openalpr import parse_annotation as parse_openalpr

print(f"REPO_ROOT = {REPO_ROOT}")

## Каталог датасетов

In [ ]:
@dataclass(frozen=True)
class DatasetSource:
    name: str
    kind: str           # "ccpd" | "yolo_bbox" | "openalpr"
    images_dir: Path
    labels_dir: Path | None = None  # None для CCPD (всё в имени)

    @property
    def n_images(self) -> int:
        return sum(1 for _ in self.images_dir.glob("*.[jp][pn]g"))


DATA = REPO_ROOT / "data"

DATASETS: list[DatasetSource] = [
    # CCPD subset-ы — у каждого свой набор edge cases
    DatasetSource("CCPD base", "ccpd", DATA / "ccpd/CCPD2019/ccpd_base"),
    DatasetSource("CCPD blur", "ccpd", DATA / "ccpd/CCPD2019/ccpd_blur"),
    DatasetSource("CCPD challenge", "ccpd", DATA / "ccpd/CCPD2019/ccpd_challenge"),
    DatasetSource("CCPD db", "ccpd", DATA / "ccpd/CCPD2019/ccpd_db"),
    DatasetSource("CCPD fn", "ccpd", DATA / "ccpd/CCPD2019/ccpd_fn"),
    DatasetSource("CCPD rotate", "ccpd", DATA / "ccpd/CCPD2019/ccpd_rotate"),
    DatasetSource("CCPD tilt", "ccpd", DATA / "ccpd/CCPD2019/ccpd_tilt"),
    DatasetSource("CCPD weather", "ccpd", DATA / "ccpd/CCPD2019/ccpd_weather"),
    # Roboflow
    DatasetSource("Roboflow Russian (train)", "yolo_bbox",
                  DATA / "roboflow/russian/train/images",
                  DATA / "roboflow/russian/train/labels"),
    DatasetSource("Roboflow Russian (test)", "yolo_bbox",
                  DATA / "roboflow/russian/test/images",
                  DATA / "roboflow/russian/test/labels"),
    DatasetSource("Roboflow European (train)", "yolo_bbox",
                  DATA / "roboflow/european/train/images",
                  DATA / "roboflow/european/train/labels"),
    DatasetSource("Roboflow Generic (train)", "yolo_bbox",
                  DATA / "roboflow/generic/train/images",
                  DATA / "roboflow/generic/train/labels"),
    # OpenALPR
    DatasetSource("OpenALPR br", "openalpr",
                  DATA / "openalpr_raw/endtoend/br"),
    DatasetSource("OpenALPR eu", "openalpr",
                  DATA / "openalpr_raw/endtoend/eu"),
    DatasetSource("OpenALPR us", "openalpr",
                  DATA / "openalpr_raw/endtoend/us"),
    # Свои фото — появятся когда юзер положит
    DatasetSource("Manual (own photos)", "manual",
                  DATA / "manual"),
]

# Сводка
for ds in DATASETS:
    if ds.images_dir.exists():
        print(f"  {ds.name:<32} {ds.n_images:>6} фото  ({ds.images_dir.relative_to(REPO_ROOT)})")
    else:
        print(f"  {ds.name:<32}   (нет:  {ds.images_dir.relative_to(REPO_ROOT)})")

## Парсинг и отрисовка

In [ ]:
CORNER_COLORS = ("red", "green", "blue", "yellow")  # TL, TR, BR, BL


def _parse_yolo_label_line(line: str) -> dict:
    """Распарсить строку YOLO label (bbox или bbox+keypoints).

    Возвращает: {cls:int, bbox:(cx,cy,w,h) normalised, keypoints:list[(x,y,v)]|None}
    """
    parts = line.strip().split()
    if len(parts) < 5:
        return {}
    cls = int(parts[0])
    bbox = tuple(float(x) for x in parts[1:5])
    keypoints = None
    if len(parts) > 5:
        rest = parts[5:]
        # YOLO-keypoints: x,y,v на точку
        if len(rest) % 3 == 0:
            keypoints = [
                (float(rest[i]), float(rest[i + 1]), int(float(rest[i + 2])))
                for i in range(0, len(rest), 3)
            ]
    return {"cls": cls, "bbox": bbox, "keypoints": keypoints}


def _draw_yolo_bbox(ax, w, h, bbox, color="magenta", label=None):
    cx, cy, bw, bh = bbox
    x = (cx - bw / 2) * w
    y = (cy - bh / 2) * h
    rect = mpatches.Rectangle((x, y), bw * w, bh * h,
                              linewidth=2, edgecolor=color, facecolor="none")
    ax.add_patch(rect)
    if label is not None:
        ax.text(x, max(y - 4, 0), label, color=color, fontsize=8,
                bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none"))


def _draw_keypoints(ax, w, h, keypoints):
    pts = [(x * w, y * h) for x, y, _ in keypoints]
    if len(pts) == 4:
        poly = mpatches.Polygon(pts, closed=True, fill=False,
                                 edgecolor="cyan", linewidth=1.5)
        ax.add_patch(poly)
    for (x, y), color in zip(pts, CORNER_COLORS):
        ax.plot(x, y, marker="o", markersize=8, markerfacecolor=color,
                markeredgecolor="black", markeredgewidth=1)


def _render_one(ax, img_path: Path, ds: DatasetSource):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    ax.imshow(img)
    ax.set_axis_off()

    if ds.kind == "ccpd":
        try:
            sample = parse_ccpd_filename(img_path.name)
            x1, y1, x2, y2 = sample.bbox_xyxy
            ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                             linewidth=2, edgecolor="magenta",
                                             facecolor="none"))
            corners = sample.corners_clockwise  # TL, TR, BR, BL
            poly = mpatches.Polygon(list(corners), closed=True, fill=False,
                                     edgecolor="cyan", linewidth=1.5)
            ax.add_patch(poly)
            for (x, y), color in zip(corners, CORNER_COLORS):
                ax.plot(x, y, marker="o", markersize=8, markerfacecolor=color,
                        markeredgecolor="black", markeredgewidth=1)
            ax.set_title(f"{sample.plate_text}\nblur={sample.blurriness} bright={sample.brightness}",
                         fontsize=8)
        except Exception as exc:
            ax.set_title(f"parse fail: {exc}", fontsize=8, color="red")

    elif ds.kind == "yolo_bbox":
        if ds.labels_dir is None:
            ax.set_title("no labels_dir", fontsize=8)
            return
        lbl = ds.labels_dir / (img_path.stem + ".txt")
        if not lbl.exists():
            ax.set_title("no label", fontsize=8, color="orange")
            return
        for line in lbl.read_text().splitlines():
            d = _parse_yolo_label_line(line)
            if not d:
                continue
            _draw_yolo_bbox(ax, w, h, d["bbox"], label=str(d["cls"]))
            if d["keypoints"]:
                _draw_keypoints(ax, w, h, d["keypoints"])
        ax.set_title(img_path.name, fontsize=7)

    elif ds.kind == "openalpr":
        txt = img_path.with_suffix(".txt")
        if not txt.exists():
            ax.set_title("no .txt", fontsize=8, color="orange")
            return
        try:
            s = parse_openalpr(txt)
            x1, y1, x2, y2 = s.bbox_xyxy
            ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                             linewidth=2, edgecolor="magenta",
                                             facecolor="none"))
            ax.set_title(s.plate_text, fontsize=8)
        except Exception as exc:
            ax.set_title(str(exc), fontsize=7, color="red")

    else:  # manual — без разметки, просто фото
        ax.set_title(img_path.name, fontsize=7)


def show_random(ds: DatasetSource, n: int = 8, seed: int | None = None,
                cols: int = 4, figsize_per: float = 3.5):
    """Сетка с n случайными фото из датасета и нанесённой разметкой."""
    if not ds.images_dir.exists():
        print(f"❌  Нет директории: {ds.images_dir}")
        return
    files = sorted(ds.images_dir.glob("*.[jp][pn]g"))
    if not files:
        print(f"❌  Нет .jpg/.png в {ds.images_dir}")
        return
    rng = random.Random(seed)
    pool = rng.sample(files, min(n, len(files)))
    rows = (len(pool) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols,
                              figsize=(cols * figsize_per, rows * figsize_per))
    axes = axes.flatten() if rows * cols > 1 else [axes]
    for ax in axes[len(pool):]:
        ax.set_axis_off()
    fig.suptitle(f"{ds.name}  ({ds.n_images} фото всего)", fontsize=12)
    for ax, img_path in zip(axes, pool):
        _render_one(ax, img_path, ds)
    plt.tight_layout()
    plt.show()

## Интерактивный виджет

Выбери датасет, число фото, нажми **🎲 Обновить** для новой случайной выборки.

In [ ]:
_options = [(f"{d.name}  ({d.n_images} фото)" if d.images_dir.exists() else f"{d.name}  (НЕТ)", i)
            for i, d in enumerate(DATASETS)]

ds_select = widgets.Dropdown(options=_options, description="Датасет:",
                              layout=widgets.Layout(width="500px"))
n_slider = widgets.IntSlider(value=8, min=1, max=24, step=1, description="N фото:")
seed_box = widgets.IntText(value=0, description="Seed:",
                            layout=widgets.Layout(width="180px"))
btn = widgets.Button(description="🎲 Обновить", button_style="primary")
out = widgets.Output()


def _on_click(_):
    with out:
        clear_output(wait=True)
        ds = DATASETS[ds_select.value]
        seed = seed_box.value if seed_box.value != 0 else None
        show_random(ds, n=n_slider.value, seed=seed)


btn.on_click(_on_click)
display(widgets.VBox([widgets.HBox([ds_select, n_slider, seed_box, btn]), out]))
_on_click(None)

## Сводная статистика по всем датасетам

In [ ]:
import pandas as pd

rows = []
for ds in DATASETS:
    rows.append({
        "dataset": ds.name,
        "kind": ds.kind,
        "n_images": ds.n_images if ds.images_dir.exists() else 0,
        "path": str(ds.images_dir.relative_to(REPO_ROOT)),
    })
df = pd.DataFrame(rows)
total = df["n_images"].sum()
print(f"Всего фото: {total:,}")
df